# **준비**

In [ ]:
!pip install torch==2.4.0 transformers==4.45.1 datasets==3.0.1 accelerate==0.34.2 trl==0.11.1 peft==0.13.0

In [ ]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLm, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# **데이터 전처리**

In [ ]:
# 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/klue-mrc-ko-rag-dataset", split="train")

# system_message 정의
system_message = """당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다."라고 답변하십시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어 특정 문장이나
문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
------
{search_result}"""

# 원본 데이터의 type별 분포 출력
print("원본 데이터의 type 분포:")

for type_name in set(dataset['type']):
  print(f"{type_name}: {dataset['type'].count(type_name)}")

# train/test 분할 비율 성정 (0.5면 5:5로 분할)
test_train = 0.8

train_data = []
test_data = []

# type별로 순회하면서 train/test 데이터 분할
for type_name in set(dataset['type']):
  # 현재 type에 해당하는 데이터의 인덱스만 추출
  curr_type_data = [i for i in range(len(dataset)) if dataset[i]['type'] == type_name]

  # test_ratio에 따라 test 데이터 개수 계산
  test_size = int(len(curr_type_data) * test_ratio)

  # 현재 type의 데이터를 test_ratio 비율로 분할하여 추가
  test_data.extend(curr_type_data[:test_size])
  train_data.extend(curr_type_data[test_size:])

# OpenAI format으로 데이터를 변환하기 위한 함수
def format_data(sample):
  # 검색 결과를 문서1, 문서2... 형태로 포매팅
  # 리스트 마다 "\n------\n".join 작동
  search_result = "\n------\n".join([f"문서{idx + 1}: {result}" for idx, result in enumerate(sample["search_result"])])

  # OpenAI format으로 변환
  return {
      "messages": [
          {
              "role": "system",
              "content": system_message.format(search_result=search_result) # placeholder 안에 변수로 기입 - 기본제공 메서드
          },
          {
              "role": "user",
              "content": sample["question"],
          },
          {
              "role": "assistant",
              "content": sample["answer"]
          },
      ],
  }

# 분할된 데이터를 OpenAI format으로 변환
train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

# 최종 데이터셋 크기 출력
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

# 분할된 데이터의 type별 분포 출력
print("\n학습 데이터의 type 분포:")
for type_name in set(dataset['type']):
  count = sum(1 for i in train_data if dataset[i]['type'] == type_name)
  print(f"{type_name}: {count}")

print("\n테스트 데이터의 type 분포:")
for type_name in set(dataset['type']):
  count = sum(1 for i in test_data if dataset[i]['type'] == type_name)
  print(f"{type_name}: {count}")

In [ ]:
train_dataset[345]["messages"]

# **데이터 타입 변경**

In [ ]:
# 리스트 형태에서 다시 Dataset 객체로 변경
print(type(train_dataset))
print(type(test_dataset))

train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(type(train_dataset))
print(type(test_dataset))

# **모델과 토크나이저 로드**

In [ ]:
# 허깅페이스 모델 이름
model_id = "Qwen/Qwen2-7B-Instruct"

# 모델과 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    # 정밀도
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# **챗 템플릿**

In [ ]:
# 템플릿 적용
text = tokeizer.apply_chat_template(
    train_dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)

print(text)

# **LoRA 학습 설정값**

In [ ]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# **학습을 위한 설정값**

In [ ]:
args = SFTConfig(
    output_dri = "qwen2-7b-rag-ko", # 저장될 디렉터리와 저장소 ID
    num_train_epochs=3, # 학습할 총 에포크 수
    per_device_train_batch_size=2, # GPU당 배치 크기
    gradient_accumulation_stpes=2, # 그래디언트 누적 스텝 수
    gradient_checkpointing=True, # 메모리 절약을 위한 체크포인팅
    optim="adamw_torch_fused", # 최적화기
    logging_steps=10, # 로그 기록 주기
    save_strategy="steps", # 저장 전략
    save_steps=50, # 저장 주기
    bf16=True, # bfloat 16 사용
    learning_rate=1e-4, # 학습률
    max_grad_norm=0.3 # 그래디언트 클리핑
    warmup_ratio=0.03, # 워밍업 비율
    lr_scheduler_type="constant", # 고정 학습률
    push_to_hub=False, # 허깅페이스에 저장할지 여부
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=None
)

# **전처리 함수**

In [ ]:
def collate_fn(batch):
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    for example in batch:
        # messages의 각 내용에서 개행문자 제거
        clean_messages = []
        for message in example["messages"]:
            clean_message = {
                "role": message["role"],
                "content": message["content"]
            }
            clean_messages.append(clean_message)

        # 깨끗해진 메시지로 템플릿 적용
        text = tokenizer.apply_chat_template(
            clean_messages,
            tokenize=False,
            add_generation_prompt=False
        ).strip()

        # 텍스트를 토큰화
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        # 레이블 초기화
        labels = [-100] * len(input_ids)

        # assistant 응답 부분 찾기
        im_start = "<|im_start|>"
        im_end = "<|im_end|>"
        assistant = "assistant"

        # 토큰 ID 가져오기
        im_start_tokens = tokenizer.encode(im_start, add_special_tokens=False)
        im_end_tokens = tokenizer.encode(im_end, add_special_tokens=False)
        assistant_tokens = tokenizer.encode(assistant, add_special_tokens=False)

        i = 0
        while i < len(input_ids):
            # <|im_start|>assistant 찾기
            if (i + len(im_start_tokens) <= len(input_ids) and
                input_ids[i:i+len(im_start_tokens)] == im_start_tokens):

                # assistant 토큰 찾기
                assistant_pos = i + len(im_start_tokens)
                if (assistant_pos + len(assistant_tokens) <= len(input_ids) and
                    input_ids[assistant_pos:assistant_pos+len(assistant_tokens)] == assistant_tokens):

                    # assistant 응답의 시작 위치로 이동
                    current_pos = assistant_pos + len(assistant_tokens)

                    # <|im_end|>를 찾을 때까지 레이블 설정
                    while current_pos < len(input_ids):
                        if (current_pos + len(im_end_tokens) <= len(input_ids) and
                            input_ids[current_pos:current_pos+len(im_end_tokens)] == im_end_tokens):
                            # <|im_end|> 토큰도 레이블에 포함
                            for j in range(len(im_end_tokens)):
                                labels[current_pos + j] = input_ids[current_pos + j]
                            break
                        labels[current_pos] = input_ids[current_pos]
                        current_pos += 1

                    i = current_pos

            i += 1

        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)

    # 패딩 적용
    max_length = max(len(ids) for ids in new_batch["input_ids"])

    for i in range(len(new_batch["input_ids"])):
        padding_length = max_length - len(new_batch["input_ids"][i])

        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * padding_length)
        new_batch["attention_mask"][i].extend([0] * padding_length)
        new_batch["labels"][i].extend([-100] * padding_length)

    # 텐서로 변환
    for k, v in new_batch.items():
        new_batch[k] = torch.tensor(v)

    return new_batch

# **전처리 테스트**

In [ ]:
# 데이터의 최대 길이 한도를 지정. 최대 8192개의 토큰까지만 사용
max_seq_length=8192

example = train_dataset[0]
batch = collate_fn([example])

print('입력에 대한 정수 인코딩 결과:')
print(batch["input_ids"][0].tolist())
print('레이블에 대한 정수 인코딩 결과:')
print(batch["labels"][0].tolist())

# **학습 진행**

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    max_seq_length=max_seq_langth, # 최대 시퀀스 길이 설정
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

# 학습 시작
trainer.train() # 모델이 자동으로 허브와 output_dir에 저장됨

# 모델 저장
trainer.save_model() # 최종 모델을 저장